# 99 — Cleanup

## What this demonstrates

Three-step Verity cleanup contract, exercised as the inverse of `00_setup.ipynb`:

1. **Preview activity** — `GET /api/v1/applications/ds_workbench/activity` shows counts of decisions, overrides, execution contexts, and entity mappings we accumulated.
2. **Purge activity** — `DELETE /api/v1/applications/ds_workbench/activity` removes decisions, overrides, and execution contexts (guarded by `VERITY_ALLOW_PURGE=1`). Entity mappings survive because they are authoring artifacts, not run artifacts.
3. **Unregister** — `DELETE /api/v1/applications/ds_workbench` removes the entity mappings and the application row itself.

After the third step, a fresh workbench session can start with a clean slate by re-running `00_setup.ipynb`.

**Safety.** Step 2 is irreversible. `VERITY_ALLOW_PURGE=1` must be set in the Verity process environment — for local dev this is baked into `docker-compose.yml`. Without it, the API returns 400 and no rows are touched.

**Note on decision attribution.** Decisions triggered through the REST runtime endpoints are currently tagged with the server's `application='default'` (because the API host's Verity client has that identity). Those decisions are NOT caught by this app's activity purge. To remove them, use Verity's admin UI or a direct SQL delete.


## Prerequisites

The `ds_workbench` application must be registered (run `00_setup.ipynb` once first). Nothing else.


In [ ]:
import os, sys
HERE = os.getcwd()
if os.path.basename(HERE) != 'ds_workbench':
    for candidate in (os.path.dirname(HERE),
                      os.path.dirname(os.path.dirname(HERE)),
                      '/home/jovyan/work'):
        if os.path.isdir(os.path.join(candidate, 'utility')):
            sys.path.insert(0, candidate); break

from utility.verity import VerityAPI, VerityAPIError
from utility.html import inject_style, badge, render_list, render_detail, render_cards

inject_style()
v = VerityAPI(application='ds_workbench')

## Preview activity

Check what we're about to delete. If all counts are zero, step 2 (purge) is effectively a no-op but safe to run anyway — it will just return zero deletes.


In [ ]:
try:
    activity = v.get_app_activity()
    app = activity['application']
except VerityAPIError as exc:
    if exc.status == 404:
        print('ds_workbench is not registered — nothing to clean up.')
        raise SystemExit(0)
    raise

render_cards([
    ('Decisions',           activity['decision_count'],           'to be deleted'),
    ('Overrides',           activity['override_count'],           'to be deleted'),
    ('Execution contexts',  activity['execution_context_count'],  'to be deleted'),
    ('Entity mappings',     activity['entity_mapping_count'],     'survive purge, removed at unregister'),
])

## Execute — Step 1: purge activity

`DELETE /api/v1/applications/ds_workbench/activity` — wipes decisions, overrides, and execution contexts in one transactional call (override_log → agent_decision_log → execution_context, in that order, to respect FK constraints). Guarded by `VERITY_ALLOW_PURGE=1` in the Verity container.


In [ ]:
try:
    purge_result = v.purge_app_activity()
    print('Purge result:', purge_result)
except VerityAPIError as exc:
    print(f'Purge blocked (status={exc.status}):', exc.detail)

## Execute — Step 2: unregister the application

Now that activity is clear, `DELETE /api/v1/applications/ds_workbench` drops the entity mappings and the application row. If activity still remained (purge skipped or blocked), the `execution_context` FK would reject this delete and the API would surface a clear 409 with a hint to purge first.


In [ ]:
try:
    result = v.unregister_application()
    print('Unregister result:', result)
except VerityAPIError as exc:
    print(f'Unregister failed (status={exc.status}):', exc.detail)

## Review results

Confirm the app is gone and the catalog is back to its pre-workbench state.


In [ ]:
try:
    v.get_application('ds_workbench')
    print('ds_workbench still exists — unregister did not complete.')
except VerityAPIError as exc:
    if exc.status == 404:
        print('Confirmed: ds_workbench is gone.')
    else:
        raise

In [ ]:
render_list(
    v.list_applications(),
    columns=[('name','Name'), ('display_name','Display'), ('description','Description')],
    title='Remaining applications',
)

---

Cleanup complete. Re-run `00_setup.ipynb` at any time to start a fresh workbench session.
